# Requirement 5: Accept availability attacks

> Administrators may be able to stop, delete, or disrupt sessions. Those actions must cause failure, not disclosure.

**Covered by:** Caller

**How it is fulfilled:** The orchestrator treats any disruption of a session as a failure, discards the session key, and retries only in a new environment with new secrets.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
session_key = install_session_key(sandbox, session_id)
print("session", session_id, "established")

session 46ff69d9 established


In [3]:
# Someone deletes the sandbox mid-session
orchestrator().delete_sandbox(sandbox.sandbox_id)
try:
    sandbox.exec("echo still here")
    outcome = "operation succeeded"
except Exception as e:
    outcome = f"failed ({type(e).__name__})"
print("next operation:", outcome)
check("disruption produces failure, not disclosure", outcome.startswith("failed"))

next operation: failed (ResourceNotFoundError)
PASS  disruption produces failure, not disclosure


In [4]:
# The orchestrator discards the key and starts over in a new environment
del session_key
sandbox2, session_id2 = new_session()
check("a retry uses a new session id", session_id2 != session_id)
cleanup(sandbox2)

PASS  a retry uses a new session id


**Caveat:** An administrator with enough rights could still delete the sandbox. Availability is not protected; confidentiality is.